In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/used_cars_cleaned.csv")

print("Shape:", df.shape)
print(df.columns.tolist())

Shape: (61056, 16)
['name', 'year', 'fuel', 'transmission', 'registration_location', 'color', 'assembly', 'body_type', 'price_pkr', 'mileage_km', 'engine_cc', 'battery_kwh', 'feature_count', 'brand', 'model', 'vehicle_age']


In [2]:
# Annual vehicle usage
df["mileage_per_year"] = np.where(
    df["vehicle_age"] > 0,
    df["mileage_km"] / df["vehicle_age"],
    df["mileage_km"]
)

# Log-transformed target
df["log_price"] = np.log1p(df["price_pkr"])

print(df.shape)

df[
    [
        "price_pkr",
        "log_price",
        "vehicle_age",
        "mileage_km",
        "mileage_per_year"
    ]
].head()

(61056, 18)


,price_pkr,log_price,vehicle_age,mileage_km,mileage_per_year
0,2390000.0,14.686804,3,120000.0,40000.000000
1,3340000.0,15.021482,3,37110.0,12370.000000
2,630000.0,13.353477,30,786.0,26.200000
3,3325000.0,15.016981,13,133000.0,10230.769231
4,1645000.0,14.313252,27,125225.0,4637.962963


In [3]:
feature_cols = [
    "vehicle_age",
    "mileage_km",
    "mileage_per_year",
    "engine_cc",
    "feature_count",
    "fuel",
    "transmission",
    "assembly",
    "body_type",
    "brand",
    "model",
    "registration_location"
]

target_col = "log_price"

X = df[feature_cols].copy()
y = df[target_col].copy()

print("X:", X.shape)
print("y:", y.shape)
print("\nMissing values:")
print(X.isna().sum().sort_values(ascending=False))

X: (61056, 12)
y: (61056,)

Missing values:
engine_cc                472
vehicle_age                0
mileage_km                 0
mileage_per_year           0
feature_count              0
fuel                       0
transmission               0
assembly                   0
body_type                  0
brand                      0
model                      0
registration_location      0
dtype: int64


In [4]:
%pip install catboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from catboost import CatBoostRegressor

print("CatBoost imported successfully.")

CatBoost imported successfully.


In [6]:
from sklearn.model_selection import train_test_split

categorical_features = [
    "fuel",
    "transmission",
    "assembly",
    "body_type",
    "brand",
    "model",
    "registration_location"
]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.20,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)

Train: (39075, 12)
Validation: (9769, 12)
Test: (12212, 12)


In [7]:
from catboost import CatBoostRegressor

catboost_model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.05,
    depth=8,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=42,
    verbose=100,
    early_stopping_rounds=100
)

catboost_model.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

0:	learn: 0.6425980	test: 0.6428857	best: 0.6428857 (0)	total: 662ms	remaining: 16m 32s
100:	learn: 0.1296719	test: 0.1325158	best: 0.1325158 (100)	total: 20.3s	remaining: 4m 41s
200:	learn: 0.1114938	test: 0.1169532	best: 0.1169532 (200)	total: 40.1s	remaining: 4m 19s
300:	learn: 0.1026155	test: 0.1104303	best: 0.1104303 (300)	total: 60s	remaining: 3m 58s
400:	learn: 0.0972490	test: 0.1070728	best: 0.1070728 (400)	total: 1m 18s	remaining: 3m 35s
500:	learn: 0.0938289	test: 0.1052350	best: 0.1052343 (499)	total: 1m 31s	remaining: 3m 3s
600:	learn: 0.0913349	test: 0.1040854	best: 0.1040854 (600)	total: 1m 44s	remaining: 2m 36s
700:	learn: 0.0894353	test: 0.1032505	best: 0.1032505 (700)	total: 1m 57s	remaining: 2m 13s
800:	learn: 0.0878886	test: 0.1026784	best: 0.1026784 (800)	total: 2m 12s	remaining: 1m 55s
900:	learn: 0.0866746	test: 0.1023762	best: 0.1023753 (898)	total: 2m 27s	remaining: 1m 38s
1000:	learn: 0.0855472	test: 0.1020779	best: 0.1020779 (1000)	total: 2m 40s	remaining: 1m 

CatBoostRegressor(depth=8, early_stopping_rounds=100, eval_metric='MAE', iterations=1500, learning_rate=0.05, loss_function='MAE', random_seed=42, verbose=100)

In [11]:
import numpy as np

# Predict on validation set in log-price space
valid_pred_log = catboost_model.predict(X_valid)

# Absolute prediction errors in log-price space
absolute_residuals = np.abs(
    y_valid.to_numpy() - valid_pred_log
)

# 90% empirical prediction interval
INTERVAL_QUANTILE_90 = np.quantile(
    absolute_residuals,
    0.90
)

print(
    "90% residual quantile:",
    INTERVAL_QUANTILE_90
)

90% residual quantile: 0.23094508712366726


In [8]:
print("Best iteration:", catboost_model.get_best_iteration())
print("Best validation score:", catboost_model.get_best_score())

Best iteration: 1499
Best validation score: {'learn': {'MAE': 0.08064667802387028}, 'validation': {'MAE': 0.10080333328456778}}


In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred_log = catboost_model.predict(X_test)

y_test_pkr = np.expm1(y_test)
y_pred_pkr = np.expm1(y_pred_log)

mae_pkr = mean_absolute_error(y_test_pkr, y_pred_pkr)
rmse_pkr = np.sqrt(mean_squared_error(y_test_pkr, y_pred_pkr))
r2 = r2_score(y_test_pkr, y_pred_pkr)

mape = np.mean(
    np.abs((y_test_pkr - y_pred_pkr) / y_test_pkr)
) * 100

print(f"MAE (PKR): {mae_pkr:,.0f}")
print(f"RMSE (PKR): {rmse_pkr:,.0f}")
print(f"R²: {r2:.4f}")
print(f"MAPE: {mape:.2f}%")

MAE (PKR): 436,497
RMSE (PKR): 2,260,364
R²: 0.8593
MAPE: 9.93%


In [10]:
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "used_car_catboost_model.cbm"

catboost_model.save_model(str(model_path))

print(f"Model saved to: {model_path}")
print(f"File exists: {model_path.exists()}")

Model saved to: ..\models\used_car_catboost_model.cbm
File exists: True


In [12]:
from pathlib import Path
import json

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

calibration_path = (
    models_dir
    / "used_car_interval_calibration.json"
)

calibration_data = {
    "interval_quantile": 0.90,
    "absolute_log_residual_quantile": float(
        INTERVAL_QUANTILE_90
    ),
    "calibration_split": "validation",
    "model_version": "catboost-v1"
}

with open(
    calibration_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        calibration_data,
        file,
        indent=4
    )

print(
    f"Calibration saved to: {calibration_path}"
)
print(
    f"File exists: {calibration_path.exists()}"
)

Calibration saved to: ..\models\used_car_interval_calibration.json
File exists: True
